In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import Precision, Recall
from tensorflow.keras.metrics import Precision, Recall
import pickle
import matplotlib.pyplot as plt

In [ ]:
train_dir = r"d:\8MyLym\Retrain\train"
val_dir = r"d:\8MyLym\Retrain\val"
in_model_path = r"d:\8MyLym\PhenoBIC_QuPath_testing\PhenoBIC_model1.keras"
model_savepath = r"c:\Users\Waluigi\Downloads\PhenoBIC_retrained.keras"

In [ ]:
# Create an ImageDataGenerator for the training data with augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,
    horizontal_flip=True,
    vertical_flip=True,
)

# Create an ImageDataGenerator for the validation and test data (only rescaling)
val_test_datagen = ImageDataGenerator(rescale=1./255)

# Create the train generator
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(48, 48),    # Resize images to 48x48 pixels
    batch_size=32,
    class_mode='binary'
)

# Create the validation generator
val_generator = val_test_datagen.flow_from_directory(
    val_dir,
    target_size=(48, 48),    # Resize images to 48x48 pixels
    batch_size=32,
    class_mode='binary'
)

# Print the class indices to ensure they match your expected class_to_idx mapping
print(train_generator.class_indices)
print(val_generator.class_indices)

In [ ]:
# Load the model
model = tf.keras.models.load_model(
    in_model_path,
    compile=False,
)

In [ ]:
# Unfreeze all weights 
for layer in model.layers:
    layer.trainable = True

# Optional: check which layers are trainable
for i, layer in enumerate(model.layers):
    print(i, layer.name, layer.trainable)

In [ ]:
model.summary(expand_nested=True)

In [ ]:
# Dice loss function
def dice_loss(y_true, y_pred, smooth=1e-6):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    # Flatten arrays
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])

    if y_pred_f.shape !=  y_true_f.shape:
        raise ValueError(f"y_pred should have shape (batch_size, 1). Got {y_pred.shape}")

    
    # Compute intersection and union
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    union = tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f)
    
    dice = (2.0 * intersection + smooth) / (union + smooth)
    return 1.0 - dice  # minimize 1-Dice

In [ ]:
# Compile model
model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss=dice_loss,
    metrics=['accuracy',
        Precision(name='precision'),
        Recall(name='recall')
    ],
)

In [ ]:
# Train the model
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=100,  # Number of epochs to train for
    steps_per_epoch=len(train_generator),
    validation_steps=len(val_generator),
)
